# 03 — Hashing and Binary Search

## Why This Matters for DSA
These two techniques solve the same broad problem — "find something fast" — from opposite ends. Hashing gives up ordering entirely in exchange for O(1) average lookup, with no requirement that the data be sorted. Binary search demands sortedness (or an equivalent monotonic structure) but turns that structure into O(log n) search, using far less memory than a hash table. Knowing both cold, and knowing which one a problem is actually asking for, resolves a huge fraction of "how do I make this faster" questions you'll hit for the rest of this course.

## Prerequisites
- `03_STL_Deep_Dive` (Phase 00) — `unordered_map`/`unordered_set` basics, `map`'s `lower_bound`/`upper_bound`
- `01_Complexity_Analysis` (Phase 01) — amortized analysis (hash table resizing mirrors `vector`'s doubling argument)

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain what actually happens inside a hash table on insert/lookup, and why average-case O(1) coexists with worst-case O(n)
- Apply the frequency/grouping hashing pattern and the complement-lookup hashing pattern to new problems
- Write a binary search from scratch without off-by-one or overflow bugs, using a consistent template
- Extend binary search to find the first/last occurrence of a value, not just any occurrence
- Recognize "binary search on the answer" problems — where the search space is a range of possible answers, not array indices — and apply the monotonic-predicate technique
- Adapt binary search to work on a rotated sorted array
- Judge whether a new problem calls for hashing or binary search when both seem plausible

## Checklist Before Moving On
- [ ] I can explain hash table collisions and why load factor triggers a resize
- [ ] I can write the frequency-counting and complement-lookup hashing patterns from memory
- [ ] I can write bulletproof binary search (`lo <= hi`, overflow-safe `mid`, correct `lo`/`hi` updates) without looking it up
- [ ] I can adapt the basic template to find the first or last occurrence of a value
- [ ] I can identify when a problem is "binary search on the answer" even though it doesn't mention a sorted array at all
- [ ] I can explain the one extra decision rotated-array binary search adds to the basic template

## Table of Contents
1. Hash Tables — How They Actually Work
2. Hashing Pattern — Frequency Counting and Grouping
3. Hashing Pattern — Complement Lookups and Membership
4. Binary Search — The Bulletproof Template
5. Binary Search — Find First and Last Occurrence
6. Binary Search on the Answer
7. Binary Search on a Rotated Sorted Array
8. Hashing vs Binary Search — Choosing the Right Tool
9. Phase Checkpoint, Cheat Sheet, and Answer Key
10. LeetCode Practice Problems


## 1. Hash Tables — How They Actually Work

### The Why
"`unordered_map` is O(1)" is a fact everyone repeats — knowing WHY, and when it stops being true, is what lets you predict when a hashing-based solution will actually perform well versus when an adversarial input could degrade it.

### Core Mechanism
- Every hash table operation follows the same three conceptual steps: **hash** the key (run it through a hash function producing an integer), **bucket** it (that integer modulo the table's current capacity picks a slot), and **resolve collisions** if another key already landed in the same slot (commonly: each bucket holds a small chain of entries).
- **Why average O(1):** with a reasonable hash function, keys spread roughly evenly across buckets — each bucket ends up with a small, roughly constant number of entries, so hashing straight to the right bucket and scanning its short chain is fast regardless of how many total entries are stored.
- **Why worst case is O(n):** if many keys collide into the SAME bucket — a poor hash function, or an adversarially crafted input designed to cause collisions — that bucket's chain grows long, and operations on it degrade toward a linear scan.
- **Load factor** (entries ÷ bucket count) is what triggers an automatic resize once it gets too high — more buckets get allocated, and every key gets re-hashed and redistributed. This is the same "amortized cost" argument as `vector`'s capacity doubling from `01_Complexity_Analysis`: individual resizes are expensive, but they're rare enough that the AVERAGE cost per operation stays O(1).

### Common Pitfalls
- Assuming hash table operations are ALWAYS O(1) regardless of input — for most practical purposes this is a safe simplification, but it's not literally true, and competitive programming problems occasionally construct adversarial inputs specifically to punish a poor hash function's worst case.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <unordered_map>
#include <string>
using namespace std;

// A hash table works in 3 conceptual steps for every operation:
// 1. HASH the key -- run it through a hash function that produces an integer
// 2. BUCKET -- take that integer modulo the table's current capacity to pick a slot
// 3. RESOLVE COLLISIONS -- if another key already hashed to the same slot, handle it
//    (most commonly: each bucket holds a small list of entries, "chaining")

int main() {
    unordered_map<string, int> wordCount;

    // every insert/lookup does hash -> bucket -> (possibly) walk a short chain
    wordCount["apple"]++;
    wordCount["banana"]++;
    wordCount["apple"]++;

    for (auto &[word, count] : wordCount) {
        cout << word << ": " << count << "\n";
    }

    // WHY O(1) AVERAGE: with a good hash function, keys spread evenly across buckets,
    // so each bucket holds a small constant number of entries on average -- lookups
    // hash directly to the right bucket, then scan a short chain.
    //
    // WHY O(n) WORST CASE: if many keys collide into the SAME bucket (a bad hash
    // function, or an adversarial input crafted to cause collisions), that bucket's
    // chain grows long, and operations on it degrade toward a linear scan.
    //
    // load factor = number of entries / number of buckets. When it gets too high,
    // the hash table automatically RESIZES (similar spirit to vector's doubling from
    // 01_Complexity_Analysis) -- more buckets, keys re-hashed and redistributed,
    // keeping the average chain length small. This resize is also where the "amortized"
    // qualifier on O(1) comes from, same reasoning as vector's amortized push_back.

    cout << "bucket_count=" << wordCount.bucket_count() << " load_factor=" << wordCount.load_factor() << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Hashing Pattern — Frequency Counting and Grouping

### The Why
This pattern shows up constantly: whenever you need to bucket items by some computed property, a hash map turns "compare everything to everything" (O(n²)) into "compute a key once per item, group by key" (close to O(n)).

### Core Mechanism
- Compute a KEY that's identical for items that should be grouped together, even though the items themselves differ. For anagrams, sorting each string's characters produces exactly this — anagrams sort to the identical string, non-anagrams don't.
- Use that key to index into a hash map whose values are lists — appending each item to the list at its computed key's slot.
- **Complexity:** sorting each string costs O(k log k) for a string of length k, done once per string — O(n · k log k) total for n strings, versus an O(n² · k) brute force that would compare every pair of strings directly for the anagram relationship.

### Common Pitfalls
- Using the ORIGINAL string as the grouping key instead of a normalized/canonical form (like the sorted version) — this fails to group anything at all, since two anagrams are, by definition, different strings.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <string>
#include <unordered_map>
#include <algorithm>
using namespace std;

// FREQUENCY / GROUPING PATTERN: use a hash map to bucket items by some computed KEY --
// anagrams share the same sorted-character key, even though the original strings differ.
vector<vector<string>> groupAnagrams(vector<string>& strs) {
    unordered_map<string, vector<string>> groups;

    for (string &s : strs) {
        string key = s;
        sort(key.begin(), key.end());   // anagrams sort to the IDENTICAL string --
                                          // that sorted string is the natural grouping key
        groups[key].push_back(s);
    }

    vector<vector<string>> result;
    for (auto &[key, group] : groups) {
        result.push_back(group);
    }
    return result;
}

int main() {
    vector<string> strs{"eat", "tea", "tan", "ate", "nat", "bat"};
    auto groups = groupAnagrams(strs);

    for (auto &g : groups) {
        cout << "{ ";
        for (auto &s : g) cout << s << " ";
        cout << "}\n";
    }
    // expected 3 groups: {eat,tea,ate}, {tan,nat}, {bat}

    // complexity: sorting each string is O(k log k) where k is string length, done for
    // all n strings -- O(n * k log k) total, versus a naive O(n^2 * k) approach checking
    // every pair for the anagram relationship directly. The hash map turns "compare
    // everything to everything" into "compute a key once, group by key"

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Hashing Pattern — Complement Lookups and Membership

### The Why
This is the pattern behind Two Sum on an UNSORTED array, and it's worth explicitly contrasting with `01_Arrays_and_Two_Pointers`' sorted-array version — same underlying question, different tool, different trade-off.

### Core Mechanism
- For each element, ask "does the value I need to PAIR with this one already exist?" A hash map answers that in O(1), turning what would be a nested-loop check into a single pass.
- **Two Sum (unsorted):** for each `nums[i]`, compute the complement (`target - nums[i]`) and check whether it's already been seen. Record the current value AFTER checking, not before — this avoids incorrectly matching an element against itself when `target == 2 * nums[i]`.
- **Membership / duplicate detection:** a hash set answers "have I seen this value before?" in O(1) — checking every element against a running set as you go, rather than comparing every pair directly.
- **The direct trade-off against two pointers:** `01_Arrays_and_Two_Pointers`' sorted-array Two Sum gets O(n) time with O(1) extra space, but REQUIRES the array to already be sorted (or accepts paying O(n log n) to sort it first). This hashing version gets O(n) time on an UNSORTED array, at the cost of O(n) extra space instead of O(1). Neither is universally "better" — it depends on whether the array is already sorted and whether the extra memory is affordable.

### Common Pitfalls
- Recording the current value in the map BEFORE checking for its complement — this can incorrectly report a match where an element pairs with itself, which is wrong unless the problem explicitly allows using the same element twice.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <unordered_map>
#include <unordered_set>
using namespace std;

// COMPLEMENT LOOKUP PATTERN: for each element, ask "does the value I NEED to pair with
// this one already exist?" -- a hash map turns that question into O(1), avoiding the
// nested loop that checking every pair would require.
vector<int> twoSum(vector<int>& nums, int target) {
    unordered_map<int, int> seenValueToIndex;   // value -> index where it was seen

    for (int i = 0; i < (int)nums.size(); i++) {
        int complement = target - nums[i];
        if (seenValueToIndex.count(complement)) {
            return {seenValueToIndex[complement], i};   // found the pair
        }
        seenValueToIndex[nums[i]] = i;   // record AFTER checking -- avoids matching an
                                           // element against itself when target == 2*nums[i]
    }
    return {-1, -1};
}

// membership / duplicate detection: does this array contain any duplicate?
bool containsDuplicate(vector<int>& nums) {
    unordered_set<int> seen;
    for (int x : nums) {
        if (seen.count(x)) return true;   // O(1) membership check
        seen.insert(x);
    }
    return false;
}

int main() {
    vector<int> nums{2, 7, 11, 15};
    auto result = twoSum(nums, 9);
    cout << "two sum indices: " << result[0] << ", " << result[1] << "\n";

    vector<int> dups{1, 2, 3, 1};
    cout << "contains duplicate: " << containsDuplicate(dups) << "\n";

    vector<int> noDups{1, 2, 3, 4};
    cout << "contains duplicate: " << containsDuplicate(noDups) << "\n";

    // both are O(n) single-pass with O(n) extra space -- compare to the O(n^2) brute
    // force of checking every pair directly. This is the UNSORTED-array counterpart to
    // 01_Arrays_and_Two_Pointers' Two Sum II, which needed sortedness for its O(n)
    // two-pointer solution -- hashing gets the same O(n) time WITHOUT needing the array
    // sorted at all, at the cost of O(n) extra space instead of O(1)

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Binary Search — The Bulletproof Template

### The Why
Binary search is conceptually simple but notoriously easy to get subtly wrong — infinite loops, off-by-one misses, integer overflow. This section is about building ONE template you trust completely, so you stop re-deriving it (and re-introducing bugs) every time.

### Core Mechanism
- **Loop condition `lo <= hi`, not `lo < hi`:** with `<`, a single-element remaining range where `lo == hi` would never get checked — silently missing a target that's exactly at that position. (Section 6's "binary search on the answer" uses `<` deliberately for a different reason — know which template you're using and why.)
- **`mid = lo + (hi - lo) / 2`, not `(lo + hi) / 2`:** the second form can integer-overflow if `lo` and `hi` are both large (their sum exceeds what the integer type can hold) — the first form can't, since `hi - lo` is always a small, safe value.
- **`lo`/`hi` updates, derived from what `mid` just told you, not guessed:** if `nums[mid] < target`, EVERYTHING from `lo` to `mid` inclusive is confirmed too small, so the new range starts at `mid + 1` — never re-include `mid`, it's already been checked and ruled out. The symmetric logic applies to `hi = mid - 1`.
- When the loop ends (`lo` has crossed `hi`), the search space is exhausted — the target isn't present.

### Common Pitfalls
- Writing `lo = mid` or `hi = mid` (forgetting the `+1`/`-1`) — this can cause an infinite loop, since `mid` might not change between iterations if `lo`/`hi` doesn't shrink.
- Reusing `(lo + hi) / 2` out of habit from other languages or old muscle memory — safe for small arrays, a real bug for large enough index ranges.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// THE BULLETPROOF BINARY SEARCH TEMPLATE. The two details that cause almost every
// binary search bug: (1) `lo <= hi` vs `lo < hi` as the loop condition, and (2) how
// `mid` is computed (overflow risk), and (3) exactly how lo/hi update. Get these three
// details right, consistently, and binary search stops being error-prone.
int binarySearch(vector<int>& nums, int target) {
    int lo = 0, hi = (int)nums.size() - 1;

    while (lo <= hi) {          // <=, NOT < -- with <, a single-element range [lo,hi] where
                                  // lo==hi would never be checked, silently missing it
        int mid = lo + (hi - lo) / 2;   // NOT (lo+hi)/2 -- that can overflow if lo and hi
                                          // are both large; this formula can't overflow

        if (nums[mid] == target) {
            return mid;
        } else if (nums[mid] < target) {
            lo = mid + 1;        // target is bigger -- discard mid and everything before it
        } else {
            hi = mid - 1;        // target is smaller -- discard mid and everything after it
        }
    }

    return -1;   // lo has crossed hi -- exhausted the search space, target isn't present
}

int main() {
    vector<int> nums{1, 3, 5, 7, 9, 11, 13};

    for (int target : {7, 1, 13, 4}) {
        cout << "search(" << target << ") = " << binarySearch(nums, target) << "\n";
    }
    // expected: 7->3, 1->0, 13->6, 4->-1

    // ALWAYS re-derive lo/hi updates from what mid just told you, don't guess:
    // nums[mid] < target means EVERYTHING from lo to mid (inclusive) is too small,
    // so the new search range starts at mid+1 -- never re-include mid, it's already
    // been checked and ruled out

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Binary Search — Find First and Last Occurrence

### The Why
Plain binary search finds "a" match, with no guarantee of which one if duplicates exist. Many problems specifically need the FIRST or LAST occurrence — the fix is a small, deliberate change to what happens the moment a match is found.

### Core Mechanism
- Instead of returning immediately on a match, treat it as a CANDIDATE answer, and keep narrowing the search in one specific direction to see if an even better (earlier or later) match exists.
- **Find first occurrence:** on a match, record it, then search LEFT (`hi = mid - 1`) for an even earlier one.
- **Find last occurrence:** on a match, record it, then search RIGHT (`lo = mid + 1`) for an even later one.
- This is exactly what `std::lower_bound`/`std::upper_bound` (`03_STL_Deep_Dive`) do internally — implementing it by hand once is what makes those STL functions stop feeling like black boxes; in real code, prefer the STL versions over hand-rolling this yourself.

### Common Pitfalls
- Returning immediately on the first match found (the plain-template behavior) when the problem specifically asks for the FIRST or LAST occurrence — plain binary search gives no guarantee about which occurrence you land on when duplicates exist.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// FIND FIRST/LAST OCCURRENCE: a variant of binary search that doesn't stop the moment
// it finds a match -- it keeps narrowing to find the LEFTMOST (or RIGHTMOST) match,
// by treating a match as "still a candidate, but keep looking in one direction."
int findFirst(vector<int>& nums, int target) {
    int lo = 0, hi = (int)nums.size() - 1;
    int result = -1;

    while (lo <= hi) {
        int mid = lo + (hi - lo) / 2;
        if (nums[mid] == target) {
            result = mid;      // record this as a candidate...
            hi = mid - 1;      // ...but keep searching LEFT for an even earlier occurrence
        } else if (nums[mid] < target) {
            lo = mid + 1;
        } else {
            hi = mid - 1;
        }
    }
    return result;
}

int findLast(vector<int>& nums, int target) {
    int lo = 0, hi = (int)nums.size() - 1;
    int result = -1;

    while (lo <= hi) {
        int mid = lo + (hi - lo) / 2;
        if (nums[mid] == target) {
            result = mid;      // record this as a candidate...
            lo = mid + 1;      // ...but keep searching RIGHT for an even later occurrence
        } else if (nums[mid] < target) {
            lo = mid + 1;
        } else {
            hi = mid - 1;
        }
    }
    return result;
}

int main() {
    vector<int> nums{5, 7, 7, 7, 8, 8, 10};

    cout << "first(7) = " << findFirst(nums, 7) << "\n";   // expected 1
    cout << "last(7)  = " << findLast(nums, 7) << "\n";    // expected 3
    cout << "first(8) = " << findFirst(nums, 8) << "\n";   // expected 4
    cout << "last(8)  = " << findLast(nums, 8) << "\n";    // expected 5
    cout << "first(9) = " << findFirst(nums, 9) << "\n";   // expected -1 (not present)

    // this is exactly what std::lower_bound / std::upper_bound (03_STL_Deep_Dive) do
    // for you already -- worth implementing once by hand so the built-in versions stop
    // feeling like magic, but reach for the STL versions in real code

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Binary Search on the Answer

### The Why
This is the single most valuable generalization of binary search in this notebook — recognizing that binary search doesn't require a literal sorted ARRAY at all. It only requires a MONOTONIC PREDICATE over a range of candidate answers: "is this candidate good enough?" flips from false to true (or true to false) exactly once as the candidate increases. That's enough structure to binary search directly over the space of possible ANSWERS.

### Core Mechanism
- Identify the range of possible answers (not array indices) — in Koko Eating Bananas, the possible eating speeds, from 1 up to the largest pile.
- Write a feasibility check: "can this candidate answer actually work?" (`canFinish`, checking whether all piles can be eaten within `h` hours at a given speed).
- **Confirm monotonicity before trusting this technique:** if a slower speed works, every faster speed also works (never worse) — this guarantees a single threshold, with everything below it infeasible and everything at-or-above it feasible. Binary search converges on that exact threshold.
- **Note the loop condition is `lo < hi` here, not `lo <= hi`** — this variant converges `lo` and `hi` onto the same single value (the answer itself) rather than searching for a specific index that might not exist. When `canFinish` succeeds at `mid`, `mid` remains a valid candidate (don't discard it — `hi = mid`, not `mid - 1`), since an even smaller value might still work.
- **Complexity:** O(n · log(range of possible answers)) — each of the O(log(range)) binary search steps costs O(n) to check feasibility. Compare to trying every possible answer one at a time, which would be O(n · range) — binary search turns a linear scan over the ANSWER SPACE into a logarithmic one, the same fundamental win as ordinary binary search, just applied to "which answer is correct" instead of "which array index holds value X."

### Common Pitfalls
- Failing to verify the predicate is actually monotonic before applying this technique — if "works" and "doesn't work" can alternate unpredictably as the candidate increases, there's no single threshold, and binary search will silently converge on a WRONG answer rather than error out.
- Using `hi = mid - 1` after a successful check (like the plain search-for-a-value template) — this can inappropriately discard the correct answer itself, since `mid` might BE the minimum feasible value and needs to stay in the search space, not get excluded.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <algorithm>
using namespace std;

// BINARY SEARCH ON THE ANSWER: binary search doesn't require a literal sorted ARRAY --
// it only requires a MONOTONIC PREDICATE over a range of candidate answers ("is this
// candidate answer good enough?" flips from false to true exactly once as the candidate
// increases). Search the space of possible ANSWERS directly, using the predicate as
// the comparison, instead of searching array indices.

// Koko Eating Bananas: find the minimum eating speed k such that Koko can eat all
// piles within h hours. "Can Koko finish within h hours at speed k?" is monotonic in
// k -- if a slower speed works, all faster speeds also work (never worse), so there's
// a single threshold speed, and everything at or above it is feasible.
bool canFinish(vector<int>& piles, int h, int speed) {
    long long hoursNeeded = 0;
    for (int pile : piles) {
        hoursNeeded += (pile + speed - 1) / speed;   // ceiling division: hours to eat this pile
    }
    return hoursNeeded <= h;
}

int minEatingSpeed(vector<int>& piles, int h) {
    int lo = 1, hi = *max_element(piles.begin(), piles.end());   // search space: possible SPEEDS

    while (lo < hi) {                 // note: < here, not <=  -- converging to a single answer,
                                        // not searching for a specific missing-or-found value
        int mid = lo + (hi - lo) / 2;
        if (canFinish(piles, h, mid)) {
            hi = mid;                  // mid works -- an even slower speed might also work,
                                        // so keep mid as a candidate, search the lower half
        } else {
            lo = mid + 1;              // mid doesn't work -- need strictly faster, discard mid
        }
    }
    return lo;   // lo == hi, converged on the minimum feasible speed
}

int main() {
    vector<int> piles1{3, 6, 7, 11};
    cout << "min speed (h=8): " << minEatingSpeed(piles1, 8) << "\n";     // expected 4

    vector<int> piles2{30, 11, 23, 4, 20};
    cout << "min speed (h=5): " << minEatingSpeed(piles2, 5) << "\n";     // expected 30
    cout << "min speed (h=6): " << minEatingSpeed(piles2, 6) << "\n";     // expected 23

    // complexity: O(n log(max_pile)) -- each of the log(max_pile) binary search steps
    // costs O(n) to check feasibility across all piles. Compare to trying every possible
    // speed one at a time (O(max_pile * n)) -- binary search on the answer turns a linear
    // scan over the ANSWER SPACE into a logarithmic one, the same win as ordinary binary
    // search, just applied to "which answer is correct" instead of "which index has value X"

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. Binary Search on a Rotated Sorted Array

### The Why
A rotated sorted array (sorted, then spliced at some unknown pivot, like `[4,5,6,7,0,1,2]`) is no longer globally sorted — but binary search still works, with one extra decision added per step, because part of the array is always still normally ordered.

### Core Mechanism
- At every step, at least ONE of the two halves around `mid` is guaranteed to be normally sorted (the rotation point can only be in one half at a time). Compare `nums[lo]` to `nums[mid]` to figure out WHICH half is the sorted one.
- Once you know which half is sorted, check whether the target falls within that sorted half's value range — if so, search there; if not, the target must be in the other (still-rotated) half, so search there instead.
- This adds exactly one extra decision (which half is sorted) on top of the standard binary search template — everything else about narrowing `lo`/`hi` stays the same.
- **Complexity:** still O(log n) — the extra per-step decision is O(1) work, it doesn't change the number of steps needed.

### Common Pitfalls
- Assuming the array must be sorted end-to-end for binary search to apply at all — the real requirement is weaker: enough structure to make a provably correct decision about which half to search. Rotated sorted arrays still have that structure, just distributed differently than a plain sorted array.
- Getting the sorted-half comparison backwards (`nums[lo] <= nums[mid]` determines the LEFT half is sorted, not the right) — a flipped comparison here silently searches the wrong half.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// SEARCH IN A ROTATED SORTED ARRAY: the array was sorted, then rotated at some unknown
// pivot (e.g. [4,5,6,7,0,1,2]). It's no longer globally sorted, but binary search STILL
// works -- the key realization: at every step, at least ONE of the two halves around
// `mid` IS still normally sorted, and you can tell which one by comparing endpoints.
int search(vector<int>& nums, int target) {
    int lo = 0, hi = (int)nums.size() - 1;

    while (lo <= hi) {
        int mid = lo + (hi - lo) / 2;
        if (nums[mid] == target) return mid;

        if (nums[lo] <= nums[mid]) {
            // LEFT half [lo, mid] is normally sorted (no rotation point inside it)
            if (nums[lo] <= target && target < nums[mid]) {
                hi = mid - 1;   // target is within the sorted left half -- search there
            } else {
                lo = mid + 1;   // target must be in the other half
            }
        } else {
            // RIGHT half [mid, hi] is normally sorted instead
            if (nums[mid] < target && target <= nums[hi]) {
                lo = mid + 1;   // target is within the sorted right half -- search there
            } else {
                hi = mid - 1;   // target must be in the other half
            }
        }
    }
    return -1;
}

int main() {
    vector<int> nums{4, 5, 6, 7, 0, 1, 2};

    for (int target : {0, 4, 3, 6}) {
        cout << "search(" << target << ") = " << search(nums, target) << "\n";
    }
    // expected: 0->4, 4->0, 3->-1 (not present), 6->2

    // O(log n) -- same complexity as ordinary binary search, just with one extra decision
    // (which half is the "normally sorted" one) at each step before applying the usual
    // range-narrowing logic

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 8. Hashing vs Binary Search — Choosing the Right Tool

### The Why
Both techniques answer "find something fast," but they need different preconditions and offer different trade-offs — recognizing which one a new problem is actually asking for (sometimes disguised) is the practical skill this notebook builds toward.

### Core Mechanism
- **Reach for hashing when:** the data isn't sorted and sorting it would be wasteful or lose needed information (like original indices), you need O(1) average lookups, and you can afford O(n) extra space. Hashing has essentially no structural requirement on the input — it works on unsorted, unordered, arbitrary data.
- **Reach for binary search when:** the data is sorted (or can cheaply be treated as sorted, like a rotated sorted array), OR — the more valuable generalization — you can identify a MONOTONIC PREDICATE over a range of candidate answers, even without a literal array involved at all (Section 6). Binary search trades a stricter structural requirement for a major space advantage: O(1) extra space, versus hashing's O(n).
- **A fast self-check:** if a problem gives you a sorted array and asks you to find something, or if you can rephrase "find the best X" as "find the smallest/largest value where some check(X) becomes true," lean binary search. If the data has no useful order and you just need fast existence/frequency/grouping checks, lean hashing.
- The two aren't always mutually exclusive alternatives for the SAME problem — some problems are better solved by neither (see `01_Arrays_and_Two_Pointers` or `02_Sliding_Window`), and some combine both (like checking feasibility with a hash-based structure inside a binary-search-on-the-answer loop).

### Common Pitfalls
- Reaching for a hash map by default on every "search" problem without checking whether the input is already sorted — if it is, and especially if it's large, binary search's O(1) space is often the meaningfully better choice, not just an alternative with the same result.


## 9. Phase Checkpoint, Cheat Sheet, and Answer Key

### Pattern Cheat Sheet

| Pattern | Structural requirement | Complexity | Example |
|---|---|---|---|
| Hash frequency/grouping | None (works on unsorted data) | O(n) or O(n·k) with a per-item key computation | Group Anagrams |
| Hash complement lookup | None | O(n) time, O(n) space | Two Sum (unsorted), Contains Duplicate |
| Binary search (basic) | Sorted array | O(log n) | Search in a sorted array |
| Binary search (first/last) | Sorted array, possible duplicates | O(log n) | Find first/last position of element |
| Binary search on the answer | Monotonic predicate over a candidate range | O(n · log(range)) typical | Koko Eating Bananas |
| Binary search on rotated array | "Rotated but still sorted" structure | O(log n) | Search in Rotated Sorted Array |

### Checkpoint Questions
1. Why does a hash table's average-case O(1) coexist with a worst-case O(n)?
2. In the Two Sum complement-lookup pattern, why must you check for the complement BEFORE inserting the current value into the map?
3. What are the three specific details that make binary search "bulletproof," and what goes wrong if each one is done incorrectly?
4. How does finding the FIRST occurrence differ from plain binary search, mechanically?
5. What's the defining structural requirement for "binary search on the answer" to be valid, and how do you verify it before applying the technique?
6. In rotated sorted array search, what's the one extra decision added to the basic template, and why does it still preserve O(log n)?

### Answer Key
1. Average case assumes a reasonable hash function spreading keys roughly evenly across buckets, keeping each bucket's chain short. Worst case happens when many keys collide into the same bucket — a poor hash function or adversarial input — making that bucket's chain long and degrading operations on it toward a linear scan.
2. To avoid incorrectly matching an element against itself. If you insert first and then check, an element could find ITSELF as a "complement" whenever `target == 2 * nums[i]`, producing a false positive.
3. `lo <= hi` (not `<`) as the loop condition — with `<`, a valid single-element range would never get checked. `mid = lo + (hi-lo)/2` (not `(lo+hi)/2`) — the latter can integer-overflow with large indices. Correct `lo`/`hi` updates derived from the comparison (`mid+1`/`mid-1`, never re-including `mid`) — getting these backwards or omitting the `+1`/`-1` can cause wrong results or infinite loops.
4. On a match, instead of returning immediately, record it as a candidate and continue searching in ONE specific direction (left, toward earlier indices) to see if an even earlier match exists — narrowing `hi` past the current match rather than stopping.
5. The predicate ("is this candidate answer good enough?") must be monotonic — false for all candidates below some threshold, true for all candidates at or above it (or vice versa), with no alternating back-and-forth. Verify it by reasoning about the problem directly: does a "better" input value (e.g. a faster speed, a bigger budget) never make a previously-working answer stop working? If you can argue that cleanly, it's monotonic.
6. The extra decision is figuring out WHICH of the two halves around `mid` is still normally sorted (by comparing `nums[lo]` to `nums[mid]`), before applying the usual narrowing logic within whichever half is relevant. This is O(1) extra work per step, so it doesn't change the total number of steps — still O(log n).

### Mini Project
Implement `search2D(matrix, target)`: search for a target value in a matrix that's sorted along both rows AND columns (each row sorted left-to-right, each column sorted top-to-bottom, and typically each row's first element is greater than the previous row's last element).
1. First attempt: can you treat the entire matrix as one giant sorted 1D array (via index math converting a flat index to `(row, col)`) and apply the basic binary search template directly?
2. Second approach: start from the top-right corner and use the sorted-both-directions structure to eliminate a full row or column with each comparison, without needing a literal binary search at all — compare and contrast the complexity of both approaches.

This exercises: recognizing when a problem's structure supports the STANDARD binary search template versus needing a structurally different (but related) search strategy — the same "verify the structural requirement before applying the technique" instinct built throughout this notebook.


## 10. LeetCode Practice Problems

Grouped by pattern. Hints point at the pattern's key decision, not the full solution.

### Hashing — Frequency Counting and Grouping

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 49 | Group Anagrams | Medium | Section 2, directly |
| 242 | Valid Anagram | Easy | Same idea as #49, simplified to comparing two frequency counts directly |
| 347 | Top K Frequent Elements | Medium | Count frequencies with a hash map first, then use a heap (`03_STL_Deep_Dive` Section 7) to extract the top k |

### Hashing — Complement Lookup and Membership

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 1 | Two Sum | Easy | Section 3, directly |
| 217 | Contains Duplicate | Easy | Section 3, directly |
| 128 | Longest Consecutive Sequence | Medium | Put everything in a hash set, then only START counting a sequence from a number whose predecessor (`n-1`) is NOT in the set — this avoids re-counting the same sequence from the middle |
| 49 | (see above, also fits here) | | |

### Binary Search — Basic and Variants

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 704 | Binary Search | Easy | Section 4, directly |
| 34 | Find First and Last Position of Element in Sorted Array | Medium | Section 5, directly |
| 35 | Search Insert Position | Easy | A close cousin of `lower_bound` — what index would the target be inserted at to keep the array sorted? |
| 74 | Search a 2D Matrix | Medium | This notebook's Mini Project, approach 1 (treat as one flat sorted array via index math) |
| 240 | Search a 2D Matrix II | Medium | This notebook's Mini Project, approach 2 (start from a corner, eliminate a row or column per step) |

### Binary Search on the Answer

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 875 | Koko Eating Bananas | Medium | Section 6, directly |
| 1011 | Capacity To Ship Packages Within D Days | Medium | Same shape as Koko — binary search on possible ship capacities, feasibility check is "can all packages ship within D days at this capacity" |
| 410 | Split Array Largest Sum | Hard | Binary search on the possible "largest subarray sum," feasibility check is "can the array be split into k or fewer parts each with sum <= this candidate" |

### Binary Search on Rotated / Modified Sorted Arrays

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 33 | Search in Rotated Sorted Array | Medium | Section 7, directly |
| 153 | Find Minimum in Rotated Sorted Array | Medium | Similar "which half is sorted" reasoning as Section 7, but the target is implicitly "the rotation point" rather than a given value |
| 81 | Search in Rotated Sorted Array II | Medium | Same as #33, but duplicates can make it impossible to tell which half is sorted in some cases — think about what extra check handles that ambiguity |

### Self-Check Before Moving to `04_Sorting_Algorithms`
If you can look at a new problem and quickly decide "hash map," "plain binary search," or "binary search on the answer" — using the negative-structure test from Section 8 (does the data have useful order, and is there a monotonic predicate available) — you've internalized the judgment call this notebook is really about. `04_Sorting_Algorithms` will round out the picture: sometimes the right first move is to CREATE the sorted structure these patterns depend on, rather than assuming it's already there.
